# Amaliyot 15 — Agent va ReAct sikli

**Mos ma'ruza:** L15 — Sun'iy intellekt agentlarini yaratish  
**Capstone natijasi:** `DocumentAssistantAgent`  
**Muhit:** asosiy qism CPU'da ishlaydi; lokal Qwen bo'limi uchun Kaggle GPU va Internet kerak. API kaliti kerak emas.

Bugun tayyor agent kodini ko'chirmaymiz. Uni bosqichma-bosqich quramiz:

1. kichik NLP vositalari;
2. vosita reyestri;
3. qoidaviy rejalashtiruvchi;
4. ReAct boshqaruv sikli;
5. guardrails va capstone klassi.

**Taxminiy vaqt:** kirish 6 min · toollar 12 min · planner 12 min · ReAct 18 min · capstone 12 min · lokal Qwen 15 min · yakun 5 min.

## 0. Muhit

Asosiy lab faqat Python standart kutubxonasidan foydalanadi. Shu sababli Kaggle'da paket o'rnatish yoki internetni yoqish shart emas.

In [24]:
import json
import platform
import re
from typing import Optional

print("Python:", platform.python_version())
print("Asosiy qism tayyor. Lokal Qwen bo'limida GPU kerak bo'ladi.")

Python: 3.11.15
Asosiy qism tayyor. Lokal Qwen bo'limida GPU kerak bo'ladi.


## 1. Agent nimani boshqaradi?

Oddiy pipeline qadamlari oldindan belgilangan. Agent esa so'rov va oldingi kuzatuvlarga qarab keyingi vositani tanlaydi.

Agentning boshqaruv sikli

Bu notebookda haqiqiy LLM o'rniga tushunarli **qoidaviy planner** ishlatamiz. Shunda agent siklining mexanikasi API kalitisiz ko'rinadi. Oxirida planner qismini lokal Qwen3 modeli bilan almashtiramiz.

## 2. Agent ishlata oladigan vositalar

Tool — agent chaqira oladigan, bitta aniq vazifani bajaruvchi funksiya. Avval uchta kichik backend tayyorlaymiz: sentiment, qidiruv va qisqartirish.

Tokenizatsiya yordamchi qadamdir; u agent mavzusi emas, shuning uchun juda sodda qoldirilgan.

In [25]:
def tokenize(text: str) -> list[str]:
    """O'zbekcha apostroflarni saqlagan holda sodda tokenizatsiya."""
    return re.findall(r"[a-zA-Z0-9_ʻʼ’'-]+", text.lower())

### 2.1 Sentiment vositasi

Bu yerda maqsad kuchli klassifikator yaratish emas. Agent chaqira oladigan, kirish va chiqishi aniq funksiyani olish kifoya. Keyinchalik shu funksiyaning o'rniga oldingi capstone klassifikatorini ulash mumkin.

In [26]:
def sentiment_classify(text: str) -> dict:
    """Kichik demo sentiment vositasi."""
    tokens = set(tokenize(text))
    positive_words = {"yaxshi", "ajoyib", "zo'r", "foydali", "qulay"}
    negative_words = {"yomon", "qiyin", "sifatsiz", "muammo", "sekin"}

    positive_count = len(tokens & positive_words)
    negative_count = len(tokens & negative_words)
    if positive_count > negative_count:
        return {"label": "ijobiy", "confidence": 0.82}
    if negative_count > positive_count:
        return {"label": "salbiy", "confidence": 0.79}
    return {"label": "neytral", "confidence": 0.55}

### 2.2 Kichik bilimlar bazasi

Alohida dataset yuklamaymiz. Beshta yozuv agent va qidiruv oqimini ko'rish uchun yetarli.

In [27]:
KNOWLEDGE_BASE = [
    {"title": "NLP", "text": "NLP kompyuterga inson tilidagi matnni tahlil qilishga yordam beradi."},
    {"title": "Transformer", "text": "Transformer attention yordamida tokenlar orasidagi bog'lanishni modellashtiradi."},
    {"title": "RAG", "text": "RAG avval hujjat topadi, keyin savol va kontekstni generatorga yuboradi."},
    {"title": "Agent", "text": "Agent maqsadga qarab vosita tanlaydi, natijani kuzatadi va keyingi qadamni belgilaydi."},
    {"title": "O'zbek NLP", "text": "O'zbek tili agglutinativ bo'lgani uchun qo'shimchalar matn tahlilida muhim."},
]

`retrieve_docs()` so'rov va hujjat orasidagi umumiy tokenlarni sanaydi. Bu RAG modelining o'rnini bosmaydi; u faqat agent uchun tez va deterministik qidiruv toolidir.

In [28]:
def retrieve_docs(query: str) -> dict:
    """So'rovga eng ko'p umumiy tokeni bor hujjatni topadi."""
    query_tokens = set(tokenize(query))
    scored_documents = []
    for document in KNOWLEDGE_BASE:
        document_tokens = set(tokenize(document["text"] + " " + document["title"]))
        score = len(query_tokens & document_tokens)
        scored_documents.append((score, document))

    best_score, best_document = max(scored_documents, key=lambda item: item[0])
    return {
        "title": best_document["title"],
        "text": best_document["text"],
        "score": best_score,
    }

### 2.3 Qisqartirish vositasi

Bu ekstraktiv demo birinchi ikki gapni oladi. Agent nuqtayi nazaridan muhim narsa: funksiya matn qabul qiladi va strukturali natija qaytaradi.

In [29]:
def summarize(text: str) -> dict:
    """Birinchi ikki gapni qaytaruvchi ekstraktiv demo vosita."""
    sentences = [part.strip() for part in re.split(r"[.!?]+", text) if part.strip()]
    selected = sentences[:2]
    summary = ". ".join(selected) + ("." if selected else "")
    return {"summary": summary, "sentence_count": len(selected)}

Uchala backendni alohida sinab ko'ring. Agentni yozishdan oldin toolning o'zi ishlashiga ishonch hosil qilish debuggingni osonlashtiradi.

In [30]:
print(sentiment_classify("Bu xizmat juda yaxshi va qulay."))
print(retrieve_docs("RAG nima?"))
print(summarize("Agent vosita tanlaydi. Natijani kuzatadi. So'ng javob beradi."))

{'label': 'ijobiy', 'confidence': 0.82}
{'title': 'RAG', 'text': 'RAG avval hujjat topadi, keyin savol va kontekstni generatorga yuboradi.', 'score': 1}
{'summary': 'Agent vosita tanlaydi. Natijani kuzatadi.', 'sentence_count': 2}


## 3. Tool reyestri

Agentga faqat Python funksiyasini berish yetmaydi. Vositaning nomi, vazifasi va kutilgan kirishi ham tushunarli bo'lishi kerak.

Tool tuzilishi

Bizning qoidaviy plannerimiz `keywords` maydonidan foydalanadi. Haqiqiy LLM esa odatda `description` va argument sxemasidan ma'no chiqaradi.

In [31]:
TOOLS = {
    "sentiment_classify": {
        "function": sentiment_classify,
        "description": "Matn hissiyotini aniqlaydi.",
        "keywords": ["hissiyot", "ijobiy", "salbiy", "yaxshi", "yomon", "baho"],
    },
    "retrieve_docs": {
        "function": retrieve_docs,
        "description": "Bilimlar bazasidan mos hujjatni topadi.",
        "keywords": ["top", "qidir", "ma'lumot", "hujjat", "nima", "haqida"],
    },
    "summarize": {
        "function": summarize,
        "description": "Berilgan matnni qisqartiradi.",
        "keywords": ["xulosa", "qisqa", "qisqartir", "umumlashtir"],
    },
}

Reyestrni ko'zdan kechiring. Har bir tool bir xil shartnomaga ega: `function`, `description`, `keywords`.

In [32]:
for tool_name, tool_info in TOOLS.items():
    print(f"{tool_name:20} -> {tool_info['description']}")

sentiment_classify   -> Matn hissiyotini aniqlaydi.
retrieve_docs        -> Bilimlar bazasidan mos hujjatni topadi.
summarize            -> Berilgan matnni qisqartiradi.


## 4. Mashq 1 — keyingi vositani tanlash

Ma'ruzadagi formula:

$$a_t = \operatorname{LLM}(q, h_{t-1})$$

Biz LLM o'rniga `select_tool()` qoidaviy plannerini yozamiz. U so'rovdagi kalit so'zlarni sanaydi va tarixda ishlatilgan toolni qayta tanlamaydi.

In [33]:
def select_tool(query: str, history: list[dict], tools: dict) -> Optional[str]:
    """LLM o'rnidagi tushunarli, qoidaviy rejalashtiruvchi."""
    q = query.lower()
    used = {record["tool"] for record in history}
    best_tool, best_score = None, 0
    for name, info in tools.items():
        if name in used:
            continue
        score = sum(1 for kw in info["keywords"] if kw in q)
        if score > best_score:
            best_score, best_tool = score, name
    return best_tool if best_score > 0 else None

Tekshiruvlar uch holatni qamrab oladi: sentiment tanlash, qidiruv tanlash va tarixdagi toolni takrorlamaslik.

In [34]:
assert select_tool("Bu mahsulot yaxshi", [], TOOLS) == "sentiment_classify"
assert select_tool("RAG haqida ma'lumot top", [], TOOLS) == "retrieve_docs"

used_history = [{"tool": "sentiment_classify"}]
assert select_tool("Bu mahsulot yaxshi", used_history, TOOLS) is None
print("Mashq 1: PASS")

Mashq 1: PASS


## 5. Mashq 2 — harakatni bajarish

Planner faqat tool nomini qaytaradi. Executor shu nomni tekshiradi va mos Python funksiyasini chaqiradi:

$$o_t = \operatorname{tool}(a_t)$$

Noma'lum nomni jimgina o'tkazib yubormang; tushunarli xato chiqaring.

In [35]:
def execute_action(tool_name: str, query: str, tools: dict) -> dict:
    """Tanlangan vositani xavfsiz tekshiradi va bajaradi."""
    if tool_name not in tools:
        raise ValueError(
            f"Noma'lum vosita: {tool_name!r}. Mavjud vositalar: {list(tools)}"
        )
    return tools[tool_name]["function"](query)

Quyidagi tekshiruv haqiqiy natijani va noto'g'ri tool uchun guardrailni sinaydi.

In [36]:
observation = execute_action("sentiment_classify", "Xizmat yaxshi", TOOLS)
assert observation["label"] == "ijobiy"

try:
    execute_action("delete_everything", "test", TOOLS)
except ValueError as error:
    print("Guardrail ishladi:", error)

Guardrail ishladi: Noma'lum vosita: 'delete_everything'. Mavjud vositalar: ['sentiment_classify', 'retrieve_docs', 'summarize']


## 6. Mashq 3 — ReAct sikli

Har qadamda planner tool tanlaydi, executor uni bajaradi va natija tarixga yoziladi:

$$a_t = \operatorname{planner}(q,h_{t-1}), \qquad o_t = \operatorname{tool}(a_t)$$
$$h_t = h_{t-1} \;\Vert\; [(a_t,o_t)]$$

ReAct tarixining o'sishi

In [37]:
def format_final_answer(history: list[dict]) -> str:
    """Kuzatuvlarni foydalanuvchiga ko'rinadigan qisqa javobga aylantiradi."""
    if not history:
        return "Mos vosita topilmadi. So'rovni aniqroq yozing."
    parts = []
    for record in history:
        observation = record["observation"]
        parts.append(f"{record['tool']}: {json.dumps(observation, ensure_ascii=False)}")
    return " | ".join(parts)

`run_react()` uchta guardrailga ega bo'lishi kerak:

- `max_steps` cheksiz siklni cheklaydi;
- `None` yangi mos tool yo'qligini bildiradi;
- planner tarixdagi toolni qayta tanlamaydi.

In [38]:
def run_react(
    query: str, tools: dict, max_steps: int = 3,
    planner=select_tool,
) -> list[dict]:
    """Harakat va kuzatuvlardan iborat cheklangan ReAct sikli."""
    history = []
    for step in range(1, max_steps + 1):
        tool_name = planner(query, history, tools)
        if tool_name is None:
            break
        observation = execute_action(tool_name, query, tools)
        history.append({
            "step": step,
            "tool": tool_name,
            "input": query,
            "observation": observation,
        })
    return history

Endi bir so'rovda ikki xil niyatni yozamiz. Natijada ikki xil tool ishlashi va tarixda ikkita kuzatuv paydo bo'lishi kerak.

In [39]:
query = "Bu kurs yaxshi. RAG haqida ma'lumot ham top."
trace = run_react(query, TOOLS, max_steps=3)

assert [record["tool"] for record in trace] == [
    "retrieve_docs", "sentiment_classify"
]
print(format_final_answer(trace))

retrieve_docs: {"title": "RAG", "text": "RAG avval hujjat topadi, keyin savol va kontekstni generatorga yuboradi.", "score": 1} | sentiment_classify: {"label": "ijobiy", "confidence": 0.82}


Trace agentning kuzatiladigan ish jurnalidir. U yashirin ichki mulohazani emas, bajarilgan harakat va qaytgan kuzatuvni saqlaydi.

In [40]:
for record in trace:
    print(f"{record['step']}-qadam")
    print("  harakat :", record["tool"])
    print("  kuzatuv :", record["observation"])

1-qadam
  harakat : retrieve_docs
  kuzatuv : {'title': 'RAG', 'text': 'RAG avval hujjat topadi, keyin savol va kontekstni generatorga yuboradi.', 'score': 1}
2-qadam
  harakat : sentiment_classify
  kuzatuv : {'label': 'ijobiy', 'confidence': 0.82}


## 7. Guardrail tajribasi

Murakkab so'rov uchta toolga mos tushadi. `max_steps=2` bo'lsa, agent uchinchi harakatga o'tmasligi kerak. Bu limit sifat va xarajat orasidagi boshqariladigan kompromissdir.

In [41]:
complex_query = "Bu yaxshi matn. RAG haqida ma'lumot top va qisqa xulosa qil."
limited_trace = run_react(complex_query, TOOLS, max_steps=2)

assert len(limited_trace) == 2
print("Bajarilgan toollar:", [record["tool"] for record in limited_trace])

Bajarilgan toollar: ['retrieve_docs', 'summarize']


### Qoidaviy plannerning chegarasi

Kalit so'z ishlatilmagan parafrazda planner niyatni tushunmasligi mumkin. Bu tajriba qoidaviy routing va LLM routing orasidagi farqni ko'rsatadi.

In [42]:
paraphrase = "Ushbu yozuvning asosiy fikrlarini ixcham bayon et."
selected = select_tool(paraphrase, [], TOOLS)
print("Tanlangan tool:", selected)
print("Nega bu natija chiqdi? Qaysi tavsif yoki keywordni yaxshilash mumkin?")

Tanlangan tool: None
Nega bu natija chiqdi? Qaysi tavsif yoki keywordni yaxshilash mumkin?


## 8. Yakuniy vazifa — capstone klassi

Endi oldingi funksiyalarni bitta interfeysga yig'amiz:

```text
agent = DocumentAssistantAgent()
answer = agent.run(user_message)
trace = agent.last_trace()
```

Klass tool funksiyalarini qayta implementatsiya qilmaydi. U reyestrni qabul qiladi va bugun yozilgan `run_react()` dan foydalanadi. Keyinchalik `TOOLS` ichidagi demo funksiyalar o'rniga oldingi capstone modullarining metodlarini berish mumkin.

In [43]:
class DocumentAssistantAgent:
    """P15 capstone natijasi: planner va toollari almashtiriladigan agent."""

    def __init__(
        self, tools: Optional[dict] = None, max_steps: int = 3,
        planner=select_tool,
    ):
        self.tools = tools or TOOLS
        self.max_steps = max_steps
        self.planner = planner
        self.history = []

    def run(self, user_message: str) -> str:
        trace = run_react(user_message, self.tools, self.max_steps, self.planner)
        self.history = trace
        return format_final_answer(trace)

    def last_trace(self) -> list[dict]:
        return list(self.history)

    def reset(self) -> None:
        self.history = []

    def tool_names(self) -> list[str]:
        return list(self.tools)

Shartnoma tekshiruvi capstone klassining tashqi xulqini tekshiradi. Bu katakdan o'tsa, P15 natijasi tayyor hisoblanadi.

In [44]:
agent = DocumentAssistantAgent(max_steps=3)
answer = agent.run("Bu kurs yaxshi. Agent haqida ma'lumot top.")

assert isinstance(answer, str) and answer
assert len(agent.last_trace()) == 2
assert agent.tool_names() == list(TOOLS)
agent.reset()
assert agent.last_trace() == []
print("Capstone shartnomasi: PASS")

Capstone shartnomasi: PASS


## 9. Local LLM — Qwen3 planner

Endi faqat planner qismini lokal LLM bilan almashtiramiz. Controller, toollar va guardrails o'zgarmaydi:

```text
qoidaviy planner ─┐
                  ├─→ bir xil ReAct controller → bir xil toollar
Qwen3 planner ────┘
```

`Qwen/Qwen3-1.7B` Kaggle GPU'da lokal ishlaydi. API kaliti kerak emas, lekin birinchi yuklash uchun **Internet** va GPU kerak. Kaggle paketlari eski bo'lsa, notebook boshida faqat quyidagini o'rnating; `torch`ni qayta o'rnatmang:

```text
%pip install -q -U "transformers>=4.51" accelerate
```

**Dars uchun:** model cache'ga tushishi uchun o'qituvchi bu bo'limni mashg'ulotdan oldin bir marta ishga tushirishi ma'qul.

In [45]:
# OFFLINE_FALLBACK: True  -> CPU/oflayn rejim (lokal Qwen o'chiriladi)
#                   False -> Kaggle GPU rejimi (lokal Qwen planner ishlaydi)
# Yakuniy topshiriq talabi: Run All ikkala rejimda ham xatosiz o'tishi kerak.
import os

try:
    import torch
    _HAS_CUDA = torch.cuda.is_available()
except ImportError:
    _HAS_CUDA = False

OFFLINE_FALLBACK = (
    os.environ.get("OFFLINE_FALLBACK", "").lower() in ("1", "true")
    or not _HAS_CUDA
)

# Lokal LLM faqat GPU bor va oflayn rejim o'chiq bo'lganda yoqiladi.
USE_LOCAL_LLM = (not OFFLINE_FALLBACK) and _HAS_CUDA
LOCAL_MODEL_ID = "Qwen/Qwen3-1.7B"

print("OFFLINE_FALLBACK:", OFFLINE_FALLBACK)
print("Local LLM:", "yoqilgan" if USE_LOCAL_LLM else "o'chirilgan")
print("Model:", LOCAL_MODEL_ID)

Local LLM: yoqilgan
Model: Qwen/Qwen3-1.7B


### 9.1 Modelni bir marta yuklash

Model har bir so'rovda qayta yuklanmaydi. `float16` 1.7B modelni Kaggle T4 xotirasiga joylashtirishga yordam beradi. `device_map="auto"` mavjud GPU'ni tanlaydi.

In [46]:
if USE_LOCAL_LLM:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    assert torch.cuda.is_available(), "Kaggle Settings → Accelerator → GPU ni yoqing."
    local_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_ID)
    local_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    local_model.eval()
    print("GPU:", torch.cuda.get_device_name(0))

AssertionError: Kaggle Settings → Accelerator → GPU ni yoqing.

### 9.2 Python reyestridan tool schema yaratish

Qwen tool nomi, tavsifi va argument sxemasini ko'radi. Barcha demo toollar bitta matnli `input` qabul qiladigan umumiy adapter sifatida taqdim etiladi.

In [ ]:
def build_tool_schemas(tools: dict) -> list[dict]:
    schemas = []
    for tool_name, tool_info in tools.items():
        schemas.append({
            "type": "function",
            "function": {
                "name": tool_name,
                "description": tool_info["description"],
                "parameters": {
                    "type": "object",
                    "properties": {"input": {"type": "string"}},
                    "required": ["input"],
                },
            },
        })
    return schemas

### 9.3 Strukturali chiqishni tekshirish

Lokal model ham ba'zan formatni buzishi mumkin. Parser javob ichidan birinchi JSON obyektini oladi; noma'lum yoki takroriy toolni guardrail rad etadi.

In [ ]:
def first_json_object(text: str) -> dict:
    decoder = json.JSONDecoder()
    for position, character in enumerate(text):
        if character != "{":
            continue
        try:
            value, _ = decoder.raw_decode(text[position:])
            if isinstance(value, dict):
                return value
        except json.JSONDecodeError:
            continue
    raise ValueError(f"Qwen JSON qaytarmadi: {text[:160]}")

def validate_tool_call(text: str, history: list[dict], tools: dict):
    if "FINISH" in text.upper() and "{" not in text:
        return None
    action = first_json_object(text)
    tool_name = action.get("name")
    used_tools = {record["tool"] for record in history}
    if tool_name not in tools or tool_name in used_tools:
        raise ValueError(f"Noto'g'ri yoki takroriy tool: {tool_name}")
    return tool_name

### 9.4 Qwen bilan matn generatsiyasi

`enable_thinking=False` tool tanlashni qisqa va tez qiladi. Modeldan yashirin mulohaza emas, tool-call formatidagi qisqa javob so'raladi.

In [ ]:
def generate_qwen_action(messages: list[dict], tools: dict) -> str:
    prompt = local_tokenizer.apply_chat_template(
        messages, tools=build_tool_schemas(tools), tokenize=False,
        add_generation_prompt=True, enable_thinking=False,
    )
    model_inputs = local_tokenizer(prompt, return_tensors="pt").to(local_model.device)
    with torch.inference_mode():
        output_ids = local_model.generate(
            **model_inputs, max_new_tokens=96, do_sample=False
        )
    new_ids = output_ids[0, model_inputs["input_ids"].shape[1]:]
    generated_text = local_tokenizer.decode(new_ids, skip_special_tokens=True)
    print("Qwen chiqishi:", generated_text.strip())
    return generated_text

### 9.5 Qwen planner

Planner so'rov va oldingi kuzatuvlarni modelga beradi. So'ng strukturali chiqishni tekshirib, controller kutadigan tool nomini qaytaradi.

In [ ]:
def qwen_select_tool(query: str, history: list[dict], tools: dict):
    history_view = [
        {"tool": item["tool"], "observation": item["observation"]}
        for item in history
    ]
    messages = [
        {"role": "system", "content": (
            "Siz agent plannersiz. Bitta kerakli va ishlatilmagan toolni chaqiring. "
            "Boshqa tool kerak bo'lmasa faqat FINISH yozing."
        )},
        {"role": "user", "content": (
            f"So'rov: {query}\nOldingi kuzatuvlar: "
            f"{json.dumps(history_view, ensure_ascii=False)}"
        )},
    ]
    generated_text = generate_qwen_action(messages, tools)
    try:
        return validate_tool_call(generated_text, history, tools)
    except ValueError as error:
        print("Planner guardrail agentni to'xtatdi:", error)
        return None

### 9.6 Bir xil agent, boshqa planner

Endi capstone klassiga faqat planner funksiyasini beramiz. Qwen toolni tanlaydi; mavjud controller uni bajaradi, kuzatuvni tarixga qo'shadi va keyingi qadamni so'raydi.

In [ ]:
if USE_LOCAL_LLM:
    qwen_agent = DocumentAssistantAgent(
        planner=qwen_select_tool,
        max_steps=3,
    )
    qwen_answer = qwen_agent.run(
        "Bu kurs menga manzur bo'ldi. RAG mohiyatini ham tushuntiring."
    )
    print("\nYakuniy javob:", qwen_answer)
    print("Qadamlar:", [item["tool"] for item in qwen_agent.last_trace()])

**Taqqoslash:** qoidaviy planner faqat yozilgan keywordlarni ko'radi; Qwen esa parafraz ma'nosidan tool tanlashi mumkin. Lekin Qwen sekinroq va ba'zan noto'g'ri format qaytaradi — shu sabab parser, allowlist va `max_steps` saqlanib qoladi.

LangChain ham xuddi shu qismlarni framework sifatida boshqaradi. Bu labda lokal modelning haqiqiy tool-call formati ko'rinishi uchun `transformers` bilan bevosita ishladik.

## 10. Yakun va chiqish savollari

Bugun qurilgan zanjir:

```text
so'rov → planner → tool → observation → history → yakuniy javob
```

Muhokama qiling:

1. Nega `max_steps` agent uchun xavfsizlik chegarasi hisoblanadi?
2. Qoidaviy planner qaysi parafrazlarda xato qiladi?
3. Haqiqiy LLM tool tanlashda `description` va argument sxemasidan qanday foydalanadi?
4. `DocumentAssistantAgent` ga oldingi RAG yoki sentiment capstone modulini qanday ulardingiz?

**Chiqish bileti:** agentning oddiy pipeline'dan eng muhim farqini bir jumlada yozing.

## 11. Yakuniy topshiriq — to'liq tizim

Ushbu bo'lim yakuniy amaliy topshiriq talablarini bitta oqimda bajaradi:

1. **Kapstone modullarini import** — `capstone/modules/m15_langchain_agent.py` (Kaggle datasetdan yoki lokal repodan).
2. **Bilimlar bazasi** — `uz_kb_mini.jsonl` ni `index_kb()` orqali yuklash.
3. **Haqiqiy vositalar reyestri** — m13 sentiment + m14 RAG + summarize.
4. **5 ta sinov so'rovi** — kamida 3 xil vosita chaqirilishi (`last_trace()` bilan isbotlanadi).
5. **SentimentAPI** — `POST /predict` → `{sentiment, confidence}`.

```text
foydalanuvchi -> DocumentAssistantAgent (m15) -> planner -> [m13 | m14 | summarize] -> javob
                                   \-> SentimentAPI (capstone/app.py, FastAPI)
```

In [ ]:
# Kapstone modullarini topish: Kaggle dataset yoki lokal repo ildizi
import sys
from pathlib import Path


def _resolve_capstone_root():
    """capstone/modules/m15_langchain_agent.py joylashgan ildiz papkasini topadi."""
    # 1) Ma'lum joylar (lokal repo yoki standart Kaggle mount)
    fixed = [
        Path("/kaggle/input/uzbek-doc-assistant-capstone"),
        Path(".."),   # repo ildizi (practices/ ichidan)
        Path("."),    # joriy papka
    ]
    for p in fixed:
        if (p / "capstone" / "modules" / "m15_langchain_agent.py").exists():
            return p

    # 2) Kaggle input mountini rekursiv qidiramiz (papka joyi qanday bo'lsa ham)
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for hit in kaggle_input.rglob("m15_langchain_agent.py"):
            # .../<ildiz>/capstone/modules/m15_langchain_agent.py
            if hit.parent.name == "modules" and hit.parents[1].name == "capstone":
                return hit.parents[2]
    return None


capstone_root = _resolve_capstone_root()
assert capstone_root is not None, (
    "capstone/ topilmadi. Kaggle'da 'uzbek-doc-assistant-capstone' "
    "datasetini Add Input orqali ulang."
)
sys.path.insert(0, str(capstone_root.resolve()))

from capstone.modules.m15_langchain_agent import (
    DocumentAssistantAgent as CapstoneAgent,
)

print("Kapstone ildizi:", capstone_root.resolve())
print("m15_langchain_agent import qilindi. [OK]")

### 11.1 Bilimlar bazasi — `index_kb()`

`uz_kb_mini.jsonl` (20 hujjat, EDUCATIONAL-USE) yuklanadi va notebookdagi
`KNOWLEDGE_BASE` haqiqiy korpus bilan almashtiriladi. Fayl topilmasa, kichik
demo baza bilan oqim davom etadi — Run All hech qachon sinmaydi.

In [ ]:
KB_FILE_NAME = "uz_kb_mini.jsonl"

_kb_candidates = [
    Path("practices/d15_checkpoints") / KB_FILE_NAME,
    Path("d15_checkpoints") / KB_FILE_NAME,
    Path("../practices/d15_checkpoints") / KB_FILE_NAME,
    Path(KB_FILE_NAME),
]
if Path("/kaggle/input").exists():
    _kb_candidates.extend(Path("/kaggle/input").rglob(KB_FILE_NAME))


def index_kb() -> int:
    """uz_kb_mini.jsonl ni yuklab, KNOWLEDGE_BASE ni haqiqiy korpus bilan to'ldiradi."""
    kb_path = next((p for p in _kb_candidates if p.exists()), None)
    if kb_path is None:
        print(f"{KB_FILE_NAME} topilmadi — notebookdagi demo KNOWLEDGE_BASE ishlatiladi.")
        return len(KNOWLEDGE_BASE)

    docs = []
    with open(kb_path, encoding="utf-8") as file:
        for line in file:
            docs.append(json.loads(line))

    KNOWLEDGE_BASE.clear()
    KNOWLEDGE_BASE.extend(docs)
    print(f"Bilimlar bazasi yuklandi: {kb_path} ({len(docs)} hujjat)")
    return len(docs)


kb_size = index_kb()
assert kb_size >= 5, "Bilimlar bazasi kamida 5 hujjatdan iborat bo'lishi kerak."
print("Namuna hujjat:", KNOWLEDGE_BASE[0]["title"])

### 11.2 Haqiqiy vositalar reyestri

Demo funksiyalar o'rniga kapstone modullari ulanadi:

| Vosita | Modul | Izoh |
|--------|-------|------|
| `sentiment_classify` | m13 `FineTunedClassifier` | GPU bo'lsa BERT, aks holda TF-IDF fallback |
| `retrieve_docs` | m14 `RAGEngine` | Semantic yoki TF-IDF qidiruv; ishlamasa demo funksiya |
| `summarize` | ekstraktiv demo | m12 o'quv korpusi talab qiladi — yakuniy tizimda ekstraktiv usul qoldirildi |

Har bir vosita ishlamay qolsa demo funksiyaga qaytadi — bu MLOps'dagi
*graceful degradation* tamoyili.

In [ ]:
from capstone.modules.m13_fine_tuned_classifier import FineTunedClassifier
from capstone.modules.m14_rag_engine import RAGEngine

# m13: kichik seed-korpusda sentiment modeli (CPU da TF-IDF fallback, ~1 s)
_SEED_TEXTS = [
    "Mahsulot a'lo, narxi adolatli. Yana xarid qilaman!",
    "Yetkazib berish tez, qadoqlash mukammal. Rahmat!",
    "Zo'r mahsulot! Do'stimga ham tavsiya qildim.",
    "Xizmat juda yaxshi va qulay, sifati ajoyib.",
    "Kurs juda foydali bo'ldi, hammasi yoqdi.",
    "Sifati juda past, pul qaytarish kerak. Xafaman.",
    "Mahsulot rasmdan farqli. Aldov bu.",
    "Sotuvchi javob bermadi. Muammom hal bo'lmadi.",
    "Yetkazib berish juda sekin, qadoq yirtilgan edi.",
    "Ilova doim qotib qoladi, juda yomon ishlaydi.",
]
_SEED_LABELS = ["ijobiy"] * 5 + ["salbiy"] * 5

sentiment_model = FineTunedClassifier()
sentiment_model.fit(_SEED_TEXTS, _SEED_LABELS)


def tool_sentiment(query: str) -> dict:
    """m13 klassifikatori orqali sentiment."""
    probs = sentiment_model.predict_proba(query)
    label = max(probs, key=probs.get)
    return {"label": label, "confidence": round(probs[label], 3)}


# m14: RAG — ishlamasa (masalan, internet yo'q) demo retrieve_docs ga qaytadi
try:
    rag_engine = RAGEngine()
    rag_engine.index([doc["text"] for doc in KNOWLEDGE_BASE])

    def tool_retrieve(query: str) -> dict:
        result = rag_engine.answer(query, k=2)
        return {"answer": result["answer"][:220], "confidence": result["confidence"]}

    retrieve_backend = "m14 RAGEngine"
except Exception as error:
    print(f"RAGEngine ishga tushmadi ({error}) — demo retrieve_docs ishlatiladi.")
    tool_retrieve = retrieve_docs
    retrieve_backend = "demo retrieve_docs"

REAL_TOOLS = {
    "sentiment_classify": {
        "function": tool_sentiment,
        "description": "Matn hissiyotini m13 klassifikatori bilan aniqlaydi.",
        "keywords": ["hissiyot", "ijobiy", "salbiy", "yaxshi", "yomon", "baho"],
    },
    "retrieve_docs": {
        "function": tool_retrieve,
        "description": "Bilimlar bazasidan mos hujjat/javob topadi (m14 RAG).",
        "keywords": ["top", "qidir", "ma'lumot", "hujjat", "nima", "haqida"],
    },
    "summarize": {
        "function": summarize,
        "description": "Berilgan matnni qisqartiradi.",
        "keywords": ["xulosa", "qisqa", "qisqartir", "umumlashtir"],
    },
}

final_agent = CapstoneAgent(tools=REAL_TOOLS, max_steps=3)
print("Retrieval backend:", retrieve_backend)
print("Vositalar:", final_agent.tool_names())

### 11.3 Besh sinov so'rovi — kamida 3 xil vosita

So'rovlar `d16_checkpoints/agent_test_queries.json` dan olinadi va bitta
ko'p-bosqichli so'rov qo'shiladi. E'tibor bering: `\"Matndagi shaxslar kimlar?\"`
so'rovi ataylab hech bir vositaga mos kelmaydi — bu **guardrail** namoyishi
(agent bilmagan ishini qilishga urinmaydi).

In [ ]:
QUERIES_FILE = "agent_test_queries.json"

_query_candidates = [
    Path("practices/d16_checkpoints") / QUERIES_FILE,
    Path("d16_checkpoints") / QUERIES_FILE,
    Path("../practices/d16_checkpoints") / QUERIES_FILE,
]
if Path("/kaggle/input").exists():
    _query_candidates.extend(Path("/kaggle/input").rglob(QUERIES_FILE))

_queries_path = next((p for p in _query_candidates if p.exists()), None)
if _queries_path is not None:
    with open(_queries_path, encoding="utf-8") as file:
        test_queries = [item["query"] for item in json.load(file)]
else:
    test_queries = [
        "Bu mahsulot juda yaxshi!",
        "NLP haqida ma'lumot toping.",
        "Ushbu matnni qisqa xulosa qiling.",
        "Matndagi shaxslar kimlar?",
    ]

# 5-so'rov: bitta xabarda 3 ta vositani ishga soladigan murakkab so'rov
test_queries.append(
    "Bu kurs juda yaxshi! NLP haqida ma'lumot top va qisqa xulosa qil."
)

used_tools = set()
for number, query in enumerate(test_queries, 1):
    answer = final_agent.run(query)
    trace = final_agent.last_trace()
    used_tools.update(record["tool"] for record in trace)

    tool_chain = [record["tool"] for record in trace]
    print(f"{number}. SO'ROV   : {query}")
    print(f"   VOSITALAR: {tool_chain if tool_chain else '— (guardrail: mos vosita topilmadi)'}")
    print(f"   JAVOB    : {answer[:160]}")
    print()

assert len(test_queries) >= 5, "Kamida 5 ta sinov so'rovi bo'lishi kerak."
assert len(used_tools) >= 3, f"Kamida 3 xil vosita kerak, hozir: {used_tools}"
print(f"Ishlatilgan vositalar ({len(used_tools)} xil):", sorted(used_tools), " [OK]")

### 11.4 SentimentAPI — `POST /predict`

`capstone/app.py` dagi `create_sentiment_api()` FastAPI ilovasini yaratadi.
Kaggle tashqi port ochmaydi, shuning uchun bu yerda `TestClient` ishlatiladi —
bu xuddi HTTP so'rov yuborishday, lekin xizmatni tarmoqsiz sinaydi. Lokal
kompyuterda esa xizmatni to'liq brauzerda ko'rish mumkin:

```bash
uvicorn capstone.app:app --reload
# brauzerda: http://127.0.0.1:8000/docs
```

In [ ]:
from fastapi.testclient import TestClient
from capstone.app import create_sentiment_api

api_client = TestClient(create_sentiment_api())

# Sog'liq tekshiruvi
health = api_client.get("/health")
assert health.status_code == 200 and health.json()["status"] == "ok"
print("GET  /health  ->", health.json())

# Ijobiy va salbiy misollar
for sample_text in [
    "Mahsulot a'lo, narxi adolatli. Yana xarid qilaman!",
    "Sifati juda past, pul qaytarish kerak. Xafaman.",
]:
    response = api_client.post("/predict", json={"text": sample_text})
    assert response.status_code == 200, response.text
    payload = response.json()
    assert set(payload) == {"sentiment", "confidence"}, payload
    assert 0.0 <= payload["confidence"] <= 1.0
    print(f"POST /predict -> {payload}   ({sample_text[:35]}...)")

# Guardrail: bo'sh matn -> 422
empty_response = api_client.post("/predict", json={"text": "   "})
assert empty_response.status_code == 422
print("POST /predict (bo'sh matn) -> 422 [OK]")

### 11.5 Yakuniy tekshiruv ro'yxati

Quyidagi katak topshiriqning barcha texnik talablarini bir joyda tasdiqlaydi.
Har bir `[OK]` qatori baholash mezonlaridan biriga mos keladi.

In [ ]:
# Yakuniy shartnoma tekshiruvi
import capstone.modules.m15_langchain_agent as _m15_module

final_answer = final_agent.run("Bu kurs juda foydali va yaxshi!")
assert isinstance(final_answer, str) and final_answer
assert isinstance(final_agent.last_trace(), list)

print("[OK] run(user_message) -> str ishlaydi")
print("[OK] last_trace() -> list ishlaydi")
print(f"[OK] {len(test_queries)} ta so'rov sinovdan o'tdi (talab: >= 3)")
print(f"[OK] {len(used_tools)} xil vosita chaqirildi (talab: >= 3):", sorted(used_tools))
print("[OK] m15_langchain_agent import qilindi:", _m15_module.__file__)
print("[OK] SentimentAPI POST /predict -> {sentiment, confidence}")
print(f"[OK] Bilimlar bazasi: {kb_size} hujjat (index_kb orqali)")
print(f"[OK] Rejim: OFFLINE_FALLBACK={OFFLINE_FALLBACK}")
print()
print("Yakuniy topshiriq talablari to'liq bajarildi!")